# Explaining Mixed Clinical Data with MAIT
**Ramtin Zargari Marandi, PhD**

Continuous measurements and categorical clinical findings belong together in an explanation. This notebook runs the **Medical Artificial Intelligence Toolbox (MAIT)** on the UCI Cleveland heart-disease dataset and highlights its SHAP summary plot with categorical values shown explicitly.

**Workflow:** load the recorded data → configure MAIT → compare CatBoost and logistic regression → inspect held-out results → explore mixed-feature SHAP explanations.

[MAIT on GitHub](https://github.com/PERSIMUNE/MAIT) · [Manual](https://persimune.github.io/MAIT/MANUAL.html) · [Hugging Face](https://huggingface.co/spaces/ramtinzm/MAIT)

## 1. Set up a CPU environment
Enable **Internet** in the notebook settings. No GPU is needed. Dependencies are installed into a separate environment so the notebook does not replace Kaggle's preinstalled scientific packages. The first run takes several minutes.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, urllib.request, zipfile, hashlib
WORK = Path('/kaggle/working/mait-heart') if Path('/kaggle').exists() else Path.cwd() / 'mait-heart'
WORK.mkdir(parents=True, exist_ok=True)
ENV = Path('/kaggle/temp/mait-python311') if Path('/kaggle').exists() else WORK / 'environment'
PYTHON = ENV / 'bin/python'
# MAIT's pinned analysis dependencies use Python 3.11, independently of Kaggle's Python.
uv = shutil.which('uv')
if uv is None:
    tools = Path('/kaggle/temp/mait-tools') if Path('/kaggle').exists() else WORK / 'tools'
    tools.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen('https://pypi.org/pypi/uv/json') as response:
        metadata = json.load(response)
    wheel = next(f for f in metadata['urls'] if f['filename'].endswith('manylinux_2_17_x86_64.manylinux2014_x86_64.whl'))
    wheel_path = tools / wheel['filename']
    urllib.request.urlretrieve(wheel['url'], wheel_path)
    assert hashlib.sha256(wheel_path.read_bytes()).hexdigest() == wheel['digests']['sha256']
    with zipfile.ZipFile(wheel_path) as z:
        member = next(name for name in z.namelist() if name.endswith('/uv'))
        uv_path = tools / 'uv'
        uv_path.write_bytes(z.read(member))
    uv_path.chmod(0o755)
    uv = str(uv_path)
if not PYTHON.exists():
    subprocess.run([uv, 'venv', '--python', '3.11', str(ENV)], check=True)
subprocess.run([uv, 'pip', 'install', '--python', str(PYTHON),
                'mait-pipeline[analysis]==0.1.0a4', 'pillow'], check=True)
subprocess.run([str(PYTHON), '--version'], check=True)
print('MAIT environment ready:', PYTHON)


## 2. Load and describe the original data
The Cleveland subset contains 303 records and 13 predictors. The original outcome ranges from 0 to 4; here **0 means absence and 1–4 are combined as presence of heart disease**, following the dataset documentation. This is classification of the recorded diagnostic outcome, not prediction of future risk.

Continuous features include age, resting blood pressure, cholesterol, maximum heart rate, and ST depression. Categorical features include chest-pain type, resting ECG findings, and exercise-induced angina. We retain the recorded category meanings rather than treating their numeric codes as continuous measurements.

For a transparent introductory example, rows with missing predictors are omitted before splitting. The notebook reports how many remain; adapt missing-data handling to your own study.

**Data:** Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). *Heart Disease*. UCI Machine Learning Repository. [DOI: 10.24432/C52P4X](https://doi.org/10.24432/C52P4X), CC BY 4.0. [Variable definitions](https://archive.ics.uci.edu/dataset/45/heart+disease).


In [ ]:
import urllib.request
DATA_URL = 'https://archive.ics.uci.edu/static/public/45/heart+disease.zip'
archive = WORK / 'heart-disease.zip'
if not archive.exists():
    urllib.request.urlretrieve(DATA_URL, archive)
print('Downloaded original UCI dataset:', archive.name)


In [ ]:
PREPARE = r"""
from pathlib import Path
import zipfile, pandas as pd, json
work = Path(__file__).resolve().parent
columns = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','num']
with zipfile.ZipFile(work / 'heart-disease.zip') as z:
    member = next(n for n in z.namelist() if n.endswith('processed.cleveland.data'))
    raw = pd.read_csv(z.open(member), header=None, names=columns, na_values='?')
df = raw.dropna().copy()
mappings = {
 'sex': {0:'Female',1:'Male'},
 'cp': {1:'Typical angina',2:'Atypical angina',3:'Non-anginal pain',4:'Asymptomatic'},
 'fbs': {0:'Glucose <=120 mg/dl',1:'Glucose >120 mg/dl'},
 'restecg': {0:'Normal ECG',1:'ST-T abnormality',2:'LV hypertrophy'},
 'exang': {0:'No exercise angina',1:'Exercise angina'},
 'slope': {1:'Upsloping ST',2:'Flat ST',3:'Downsloping ST'},
 'thal': {3:'Normal perfusion',6:'Fixed defect',7:'Reversible defect'}
}
for col, labels in mappings.items():
    df[col] = df[col].map(labels)
    assert df[col].notna().all(), col
names = {'age':'Age','sex':'Sex','cp':'Chest_pain','trestbps':'Resting_BP','chol':'Cholesterol','fbs':'Fasting_glucose','restecg':'Resting_ECG','thalach':'Maximum_HR','exang':'Exercise_angina','oldpeak':'ST_depression','slope':'ST_slope','ca':'Vessel_count','thal':'Perfusion'}
df['heart_disease'] = (df.pop('num') > 0).astype(int)
df.rename(columns=names,inplace=True)
df.to_csv(work / 'cleveland.csv',index=False)
summary = {'original_rows':len(raw),'complete_rows':len(df),'excluded_rows':len(raw)-len(df),'outcome_counts':{str(k):int(v) for k,v in df.heart_disease.value_counts().items()},'categorical_features':[names[c] for c in mappings]}
(work / 'data-summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
"""
(WORK / 'prepare_data.py').write_text(PREPARE)
subprocess.run([str(PYTHON), str(WORK / 'prepare_data.py')], check=True)


## 3. Configure the real MAIT pipeline
We compare CatBoost with logistic regression. CatBoost accepts categorical predictors directly. This preserves one feature per original variable in MAIT's mixed-feature SHAP plot. We use three-fold cross-validation on the development data and a separate 20% hold-out set; hyperparameter tuning and feature selection are disabled to keep this CPU example approachable.

The following cell exports the packaged MAIT script and changes its configuration assignments. The internal configuration helper is specific to the pinned package version; the modelling and visualisation functions themselves are not rewritten. A numeric-array conversion before the later SHAP decision plot handles mixed-type output without changing its values.


In [ ]:
CONFIGURE = r"""
from pathlib import Path
import json
from mait import export_demo
from mait.export import _configure
work = Path(__file__).resolve().parent
run = work / 'analysis'
script = run / 'mait_demo.py'
if not script.exists():
    script = export_demo(run)
cat = json.loads((work / 'data-summary.json').read_text())['categorical_features']
settings = {
 'mydata': f"pd.read_csv({str(work / 'cleveland.csv')!r})",
 'cat_features': repr(cat), 'columns_to_drop': '[]',
 'models_to_include': repr(['CatBoost_mdl', 'LogisticRegression_mdl']),
 'outcome_var': repr('heart_disease'),
 'class_labels_display': repr(['Absent','Present']),
 'main_folder_name': repr(str(run / 'results')),
 'train_size_perc': '0.8', 'cv_folds': '3',
 'merged_rare_categories':'False', 'shorten_feature_names':'False',
 'fig_file_format':repr('png')
}
source = _configure(script.read_text(), settings)
# SHAP 0.47 decision plots require a numeric array after mixed-type processing.
marker = "    # Plot the SHAP decision plot with only significant features"
assert source.count(marker) == 1
source = source.replace(marker, "    shap_values = np.asarray(shap_values, dtype=float)" + chr(10) + marker)
script.write_text(source)
print('Configured MAIT script:', script)
"""
(WORK / 'configure_mait.py').write_text(CONFIGURE)
subprocess.run([str(PYTHON), str(WORK / 'configure_mait.py')], check=True)


In [ ]:
environment = os.environ.copy()
environment.update(MPLBACKEND='Agg', MPLCONFIGDIR=str(WORK / 'matplotlib-cache'),
                   OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
log = WORK / 'mait-run.log'
with log.open('w') as output:
    completed = subprocess.run([str(PYTHON), '-u', str(WORK / 'analysis/mait_demo.py')],
                               env=environment, stdout=output, stderr=subprocess.STDOUT)
print(log.read_text()[-5000:])
if completed.returncode:
    raise RuntimeError('MAIT did not finish. Inspect mait-run.log before interpreting results.')
RESULTS = WORK / 'analysis/results'
print('Analysis complete. Files:', len(list(RESULTS.iterdir())))


## 4. Inspect MAIT's mixed-feature SHAP plot
Each point represents an observation. Its horizontal position shows the feature's contribution to the model output: values to the right increase the output for the positive class; values to the left decrease it. Read the output scale in the figure rather than interpreting every SHAP value as a probability change.

**Continuous features** use a low-to-high colour scale. **Categorical features** use discrete colours, marker shapes, and a category legend, so an ECG finding or chest-pain category remains interpretable alongside blood pressure and age. Feature order reflects mean absolute SHAP contribution, not a causal effect.

The figure below explains the CatBoost cross-validation observations from the development set. Every observation is explained in its validation fold. The separate test results appear in the next section. We display MAIT's saved figure directly.


In [ ]:
from IPython.display import display, Image, FileLink
# Convert TIFF to PNG for browser display without changing the plot.
SHOW_PLOTS = sorted(RESULTS.glob('SHAP_summary_plot_CB_withcats.tif'))
assert SHOW_PLOTS, 'Expected MAIT mixed-feature SHAP figure is missing.'
convert = "from PIL import Image; import sys; Image.open(sys.argv[1]).save(sys.argv[2])"
for path in SHOW_PLOTS:
    png = path.with_suffix('.png')
    subprocess.run([str(PYTHON), '-c', convert, str(path), str(png)], check=True)
    print(path.name)
    display(Image(filename=str(png), width=1100))


## 5. Read the held-out performance results
Cross-validation results help assess the model on the development data; the ROC and confusion-matrix figure describes the separate hold-out set. The table below contains the development cross-validation results. This small historical dataset is a software demonstration, so performance should not be treated as evidence of clinical readiness. Avoid selecting settings based on repeated inspection of the test set.


In [ ]:
table = RESULTS / 'aggregated_results_all.xlsx'
if table.exists():
    export_table = "import pandas as pd,sys; print(pd.read_excel(sys.argv[1]).to_string(index=False))"
    subprocess.run([str(PYTHON), '-c', export_table, str(table)], check=True)
for path in sorted(RESULTS.glob('ROC_CM_*.png')):
    if 'dummy' not in path.name:
        display(Image(filename=str(path), width=950))


## 6. Adapt the workflow and download the outputs
For your own dataset, replace the CSV, outcome, categorical feature list, and class labels in the configuration. Start with an appropriate split for your research design: repeated observations from one person should stay together. MAIT also includes other model families, regression, and survival workflows; this notebook demonstrates binary classification.

[DAIMS](https://github.com/PERSIMUNE/DAIMS) offers dataset checks and documentation templates. [ExplaineR](https://github.com/PERSIMUNE/explainer) provides complementary model interpretation tools in R.

Download the results archive below, including the configured script, figures, model, and run log. The isolated dependency environment is excluded.


In [ ]:
import zipfile
bundle = WORK.parent / 'mait-heart-results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for root in [WORK / 'analysis']:
        for path in root.rglob('*'):
            if path.is_file(): z.write(path, path.relative_to(WORK))
    for name in ['cleveland.csv','data-summary.json','mait-run.log']:
        z.write(WORK / name, name)
display(FileLink(str(bundle)))


## References
- **MAIT:** [Source, citation, and documentation](https://github.com/PERSIMUNE/MAIT). Developed by **Ramtin Zargari Marandi, PhD**.
- **Dataset:** Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). *Heart Disease*. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X. CC BY 4.0.
- [CatBoost](https://catboost.ai/) · [SHAP](https://shap.readthedocs.io/)
